# HMAC model bake-off

Compares OCR models and AI-text detectors for HMAC on a free Colab GPU. This is the `ocr-bakeoff` and `ai-detector-bakeoff` work in [ROADMAP.md](../ROADMAP.md).

**Before you start**

1. **Runtime → Change runtime type → T4 GPU.**
2. Have the page images ready. They're git-ignored, so upload them from `API/media/coverdirectory/` on your laptop.
3. Only upload real student work with consent.

Results are saved after every page, so a dropped session loses nothing. Mount Google Drive (below) to keep them after the session ends.

In [ ]:
# GPU check and up-to-date transformers (Colab already has CUDA PyTorch).
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install -U "transformers>=5.17" accelerate

In [ ]:
# Where results go. With USE_DRIVE they survive the session (Colab asks you to sign in to Drive).
import os

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_DIR = "/content/drive/MyDrive/hmac-bakeoff"
else:
    RESULTS_DIR = "/content/hmac-bakeoff"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Results go to", RESULTS_DIR)

In [ ]:
# The repo provides the evaluation answers and the current AI-detection model (stored with Git LFS).
!apt-get -qq install -y git-lfs > /dev/null
!test -d /content/HMAC || git clone -q --depth 1 https://github.com/Hrishikesh-Panigrahi/HMAC.git /content/HMAC
!cd /content/HMAC && git lfs install --local > /dev/null && git lfs pull --include "API/new-model/*"

## Part 1: OCR

Each page lists the image you upload, what is actually written on it, and what the current pipeline (Roboflow word detection + TrOCR-base on word crops) produced. User2's page is typed in a handwriting-style font, so it's left out of the handwriting average. Add your own pages to `PAGES`.

In [ ]:
STORY = "One sunny day, Wishers decided to go on an adventure. They all decided to explore the mysterious forest at the edge of the town. With the sun casting warm rays, wishers, the adventurous cat gathered his friends. Under the green canopy of the mysterious forest, Wishers and his friends ventured forth with excitement in their hearts. The air was filled with the sweet scent of blooming flowers, and the rustle of leaves added a rhythm to their journey. As they delved deeper into the woodland, they encountered a babbling brook, its crystal-clear waters inviting them to take a refreshing pause."
FIRST_THREE = ("One sunny day, wishers decided to go on an adventure. They all decided to explore the mysterious "
               "forest at the edge of the town. With the sun casting warm rays, wishers, the adventurous cat "
               "gathered {} friends.")

PAGES = {
    "User1": {"file": "User1_Assignment.jpg", "handwritten": True, "truth": FIRST_THREE.format("their"),
              "current_pipeline": "One sunny day. Wishers decided to go on an adventure. They all decided to explore the mysterious forest at the edge of the town. With the Sun casting war on rays, wishers. The adventurous cat- Gadhered their friends."},
    "User2": {"file": "User2_Assignment.jpg", "handwritten": False, "truth": STORY,
              "current_pipeline": "One sunny day, Wishers decided to go on an adventure They all decided to explore the mysterious forest at the edge of the town with the sun casting warm rays, wishers, the adventurous cat gathered his friends. Under the green canopy of the mysterious forest, Wishers and his friends ventured forth with excitement in their hearts. The air WAS filled with the sweet scent of blooming flowers, and the rustle of leaves added a rhythm to their journey. As they delved deeper into the woodland, they encountered a babbling brook, its crystal-clear waters inviting them to take a refreshing cause."},
    "User3": {"file": "User3_Assignment.jpg", "handwritten": True, "truth": STORY,
              "current_pipeline": "One sunny day, sisters decided to go on an adventure. They all decided to explore the mysterious forest at the edge of the town. With the sun casing warm rays, wishers, the adventurous cat gathered his friends. Under the Deep Conopy of the mysterious forest, Wishers and his friends ventured forth with excitement in their hearts. The air was filled with the sweat seat of blooming flowers, and the rustle of leaves added a rhythm to their journey. As they delved deeper into the woodland, then encountered a baldding brook, its Crystal clear waters inviting them to take the refashion pause."},
    "User4": {"file": "User4_Assignment.jpg", "handwritten": True, "truth": FIRST_THREE.format("his"),
              "current_pipeline": None},  # the current pipeline never processed this page
}

In [ ]:
# Upload the page images (API/media/coverdirectory/ on your laptop). Skip this if they're already in PAGE_DIR.
from google.colab import files

PAGE_DIR = "/content/pages"
os.makedirs(PAGE_DIR, exist_ok=True)
for name, data in files.upload().items():
    with open(os.path.join(PAGE_DIR, name), "wb") as f:
        f.write(data)
missing = [p["file"] for p in PAGES.values() if not os.path.exists(os.path.join(PAGE_DIR, p["file"]))]
print("missing pages:", missing or "none")

In [ ]:
import gc
import html
import json
import re
import time

import torch
from PIL import Image

OCR_RESULTS = os.path.join(RESULTS_DIR, "ocr_results.json")
MAX_SIDE = 1600  # longest image side in pixels; lower it if a model runs out of GPU memory


def norm_words(text):
    return re.findall(r"[a-z0-9']+", (text or "").lower())


def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]


def error_rates(truth, hyp):
    """Word and character error rate, ignoring case and punctuation."""
    tw, hw = norm_words(truth), norm_words(hyp)
    tc, hc = " ".join(tw), " ".join(hw)
    return edit_distance(tw, hw) / len(tw), edit_distance(tc, hc) / len(tc)


def load_image(path):
    image = Image.open(path).convert("RGB")
    scale = MAX_SIDE / max(image.size)
    if scale < 1:
        image = image.resize((round(image.width * scale), round(image.height * scale)), Image.LANCZOS)
    return image


def strip_html(text):
    return html.unescape(re.sub(r"<[^>]+>", " ", re.sub(r"<br\s*/?>", " ", text)))


def load_results():
    return json.load(open(OCR_RESULTS)) if os.path.exists(OCR_RESULTS) else {}


def save_results(results):
    json.dump(results, open(OCR_RESULTS, "w"), indent=1)

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

# Prompts are the ones each model was trained with; don't reword them.
SURYA_PROMPT = ("OCR this image to HTML. Each block is a div with data-label and data-bbox "
                "(x0 y0 x1 y1, normalized 0-1000).")  # surya/inference/prompts.py
NANONETS_PROMPT = ("Extract the text from the above document as if you were reading it naturally. Return the tables "
                   "in html format. Return the equations in LaTeX representation. If there is an image in the "
                   "document and image caption is not present, add a small description of the image inside the "
                   "<img></img> tag; otherwise, add the image caption inside <img></img>. Watermarks should be "
                   "wrapped in brackets. Ex: <watermark>OFFICIAL COPY</watermark>. Page numbers should be wrapped "
                   "in brackets. Ex: <page_number>14</page_number> or <page_number>9/22</page_number>. Prefer "
                   "using \u2610 and \u2611 for check boxes.")  # Nanonets-OCR2 model card

MODELS = {
    "GLM-OCR": {"id": "zai-org/GLM-OCR", "prompt": "Text Recognition:"},  # 1.3B, MIT
    "Surya OCR 2": {"id": "datalab-to/surya-ocr-2", "prompt": SURYA_PROMPT, "post": strip_html},  # 0.7B
    "Nanonets OCR2 3B": {"id": "nanonets/Nanonets-OCR2-3B", "prompt": NANONETS_PROMPT,
                         "system": "You are a helpful assistant.", "post": strip_html},  # 3.8B
}


def run_ocr(name, max_new_tokens=1024):
    """Transcribe every uploaded page with one model, saving after each page."""
    spec = MODELS[name]
    results = load_results()
    done = results.setdefault(name, {})
    todo = [p for p, page in PAGES.items()
            if p not in done and os.path.exists(os.path.join(PAGE_DIR, page["file"]))]
    if not todo:
        print("nothing to do")
        return
    processor = AutoProcessor.from_pretrained(spec["id"])
    # The T4 has no bfloat16, so load in float16.
    model = AutoModelForImageTextToText.from_pretrained(
        spec["id"], dtype=torch.float16, device_map="cuda", attn_implementation="sdpa").eval()
    for p in todo:
        messages = []
        if spec.get("system"):
            messages.append({"role": "system", "content": [{"type": "text", "text": spec["system"]}]})
        messages.append({"role": "user", "content": [
            {"type": "image", "image": load_image(os.path.join(PAGE_DIR, PAGES[p]["file"]))},
            {"type": "text", "text": spec["prompt"]},
        ]})
        inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                               return_dict=True, return_tensors="pt")
        inputs = inputs.to(model.device, dtype=torch.float16)
        torch.cuda.synchronize()
        start = time.time()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        torch.cuda.synchronize()
        text = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        done[p] = {"text": spec.get("post", lambda t: t)(text).strip(), "seconds": round(time.time() - start, 1)}
        save_results(results)
        print(f"{p}: {done[p]['seconds']}s")
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()

Run the models one at a time, smallest first. If one runs out of memory, restart the runtime (results are kept), lower `MAX_SIDE`, and run its cell again.

In [ ]:
run_ocr("Surya OCR 2")

In [ ]:
run_ocr("GLM-OCR")

In [ ]:
run_ocr("Nanonets OCR2 3B")

**Optional: Chandra OCR 2** (5.3B, the largest candidate). It needs Datalab's `chandra-ocr` package, which may change installed package versions, so run it last and restart the runtime first if earlier models are still loaded.

In [ ]:
!pip -q install chandra-ocr

In [ ]:
from chandra.model.hf import generate_hf
from chandra.model.schema import BatchInputItem
from chandra.output import parse_markdown


def run_chandra():
    results = load_results()
    done = results.setdefault("Chandra OCR 2", {})
    todo = [p for p, page in PAGES.items()
            if p not in done and os.path.exists(os.path.join(PAGE_DIR, page["file"]))]
    model = AutoModelForImageTextToText.from_pretrained(
        "datalab-to/chandra-ocr-2", dtype=torch.float16, device_map="cuda").eval()
    model.processor = AutoProcessor.from_pretrained("datalab-to/chandra-ocr-2")
    model.processor.tokenizer.padding_side = "left"
    for p in todo:
        start = time.time()
        item = BatchInputItem(image=load_image(os.path.join(PAGE_DIR, PAGES[p]["file"])), prompt_type="ocr_layout")
        raw = generate_hf([item], model)[0].raw
        done[p] = {"text": parse_markdown(raw).strip(), "seconds": round(time.time() - start, 1)}
        save_results(results)
        print(f"{p}: {done[p]['seconds']}s")
    del model
    gc.collect()
    torch.cuda.empty_cache()


run_chandra()

### OCR results

Word error rate (WER) per page; lower is better. `handwritten WER` averages the real handwriting pages.

In [ ]:
import pandas as pd

results = load_results()
results["current pipeline (Roboflow + TrOCR-base)"] = {
    p: {"text": page["current_pipeline"], "seconds": None} for p, page in PAGES.items() if page["current_pipeline"]}

rows = []
for model_name, outs in results.items():
    row = {"model": model_name}
    hand = []
    for p, page in PAGES.items():
        if outs.get(p, {}).get("text"):
            wer, cer = error_rates(page["truth"], outs[p]["text"])
            row[f"{p} WER %"] = round(wer * 100, 1)
            row[f"{p} CER %"] = round(cer * 100, 1)
            if page["handwritten"]:
                hand.append(wer * 100)
    row["handwritten WER %"] = round(sum(hand) / len(hand), 1) if hand else None
    times = [o["seconds"] for o in outs.values() if o.get("seconds")]
    row["seconds / page"] = round(sum(times) / len(times), 1) if times else None
    rows.append(row)

ocr_table = pd.DataFrame(rows).set_index("model").sort_values("handwritten WER %")
ocr_table.to_csv(os.path.join(RESULTS_DIR, "ocr_summary.csv"))
ocr_table

In [ ]:
# Read the transcripts for one page side by side.
PAGE = "User3"
for model_name, outs in results.items():
    print(f"--- {model_name}\n{outs.get(PAGE, {}).get('text')}\n")

## Part 2: AI-text detectors

Labelled data:

- **Human:** answers from the HC3 dataset (Reddit "explain like I'm five" and open questions), with HC3's tokeniser spacing undone so detectors can't key on it.
- **AI:** ChatGPT answers to the same HC3 questions, plus the 20 Claude-written student answers in `API/base/evaluation_data.py`.

Every text is scored clean and again with simulated OCR errors (10% word error rate), since HMAC scores OCR output. Look at how often human text is flagged first.

In [ ]:
# Part 2 runs on its own: it only needs the setup cells at the top.
import gc
import json
import os
import random
import re
import sys
import urllib.request

import pandas as pd
import torch

sys.path.insert(0, "/content/HMAC/API")
from base.evaluation_data import QUESTIONS

LETTERS = "abcdefghijklmnopqrstuvwxyz"


def detokenize(text):
    text = re.sub(r"\s+(n't|'s|'re|'ve|'ll|'d|'m)\b", r"\1", text)
    text = re.sub(r"\s+([.,!?;:%)\]])", r"\1", text)
    text = re.sub(r"([(\[])\s+", r"\1", text)
    text = re.sub(r'"\s+(.*?)\s+"', r'"\1"', text)
    return re.sub(r"\s{2,}", " ", text).strip()


def hc3(config, count):
    human, ai, offset = [], [], 0
    while (len(human) < count or len(ai) < count) and offset < 1000:
        url = (f"https://datasets-server.huggingface.co/rows?dataset=Hello-SimpleAI/HC3&config={config}"
               f"&split=train&offset={offset}&length=100")
        for r in json.load(urllib.request.urlopen(url, timeout=60))["rows"]:
            for text in map(detokenize, r["row"]["human_answers"][:1]):
                if 60 <= len(text.split()) <= 300 and len(human) < count:
                    human.append(text)
            for text in map(detokenize, r["row"]["chatgpt_answers"][:1]):
                if 60 <= len(text.split()) <= 300 and len(ai) < count:
                    ai.append(text)
        offset += 100
    return human, ai


def ocr_noise(text, wer, rng):
    """Same model as API/base/evaluation.py: mostly misread letters, some dropped or extra words."""
    out = []
    for word in text.split():
        if rng.random() >= wer:
            out.append(word)
            continue
        kind = rng.random()
        if kind < 0.8 and len(word) > 1:
            chars = list(word)
            i = rng.randrange(len(chars))
            if chars[i].isalpha():
                chars[i] = rng.choice(LETTERS)
            out.append("".join(chars))
        elif kind < 0.95:
            continue
        else:
            out.extend([word, rng.choice(["a", "the", "it", "of"])])
    return " ".join(out)


human_eli5, ai_eli5 = hc3("reddit_eli5", 30)
human_qa, ai_qa = hc3("open_qa", 15)
SETS = {
    "human": human_eli5 + human_qa,
    "ChatGPT": ai_eli5 + ai_qa,
    "Claude student answers": [a for q in QUESTIONS.values() for a in q["answers"]],
}
rng = random.Random(0)
NOISY = {k: [ocr_noise(t, 0.10, rng) for t in v] for k, v in SETS.items()}
print({k: len(v) for k, v in SETS.items()})

In [ ]:
import torch.nn as nn
from transformers import (AutoConfig, AutoModel, AutoModelForSequenceClassification, AutoTokenizer,
                          DistilBertForSequenceClassification, DistilBertTokenizer, PreTrainedModel)


class DesklibAIDetectionModel(PreTrainedModel):
    """Model class from the desklib model card: mean-pooled DeBERTa-v3 + a linear head."""
    config_class = AutoConfig

    def __init__(self, config):
        super().__init__(config)
        self.model = AutoModel.from_config(config)
        self.classifier = nn.Linear(config.hidden_size, 1)
        self.post_init()

    def forward(self, input_ids, attention_mask=None):
        hidden = self.model(input_ids, attention_mask=attention_mask)[0]
        mask = attention_mask.unsqueeze(-1).expand(hidden.size()).float()
        return self.classifier((hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9))


def detectors():
    """Yield (name, function returning P(AI) for a text), one model at a time."""
    current = "/content/HMAC/API/new-model"
    assert os.path.getsize(f"{current}/pytorch_model.bin") > 1e6, "Git LFS didn't download the current model"
    tok = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
    model = DistilBertForSequenceClassification.from_pretrained(current, num_labels=2).cuda().eval()
    yield "current (DistilBERT)", lambda t: torch.softmax(model(**tok(t, return_tensors="pt", truncation=True, max_length=512).to("cuda")).logits, 1)[0, 1].item()

    tok = AutoTokenizer.from_pretrained("Oxidane/tmr-ai-text-detector")
    model = AutoModelForSequenceClassification.from_pretrained("Oxidane/tmr-ai-text-detector").cuda().eval()
    yield "Oxidane/tmr-ai-text-detector", lambda t: torch.softmax(model(**tok(t, return_tensors="pt", truncation=True, max_length=512).to("cuda")).logits, 1)[0, 1].item()

    tok = AutoTokenizer.from_pretrained("desklib/ai-text-detector-academic-v1.01")
    model = DesklibAIDetectionModel.from_pretrained("desklib/ai-text-detector-academic-v1.01").cuda().eval()

    def desklib(t):
        enc = tok(t, return_tensors="pt", truncation=True, max_length=768).to("cuda")
        return torch.sigmoid(model(enc["input_ids"], enc["attention_mask"]))[0, 0].item()
    yield "desklib/ai-text-detector-academic-v1.01", desklib


def auroc(human_scores, ai_scores):
    pairs = [(a > h) + 0.5 * (a == h) for a in ai_scores for h in human_scores]
    return sum(pairs) / len(pairs)


rows = []
with torch.no_grad():
    for name, score in detectors():
        for variant, data in (("clean", SETS), ("OCR 10%", NOISY)):
            s = {k: [score(t) for t in texts] for k, texts in data.items()}
            rows.append({
                "detector": name, "text": variant,
                "human flagged as AI %": round(100 * sum(x >= 0.5 for x in s["human"]) / len(s["human"]), 1),
                "ChatGPT caught %": round(100 * sum(x >= 0.5 for x in s["ChatGPT"]) / len(s["ChatGPT"]), 1),
                "Claude answers caught %": round(100 * sum(x >= 0.5 for x in s["Claude student answers"]) / len(s["Claude student answers"]), 1),
                "AUROC vs ChatGPT": round(auroc(s["human"], s["ChatGPT"]), 3),
                "AUROC vs Claude": round(auroc(s["human"], s["Claude student answers"]), 3),
            })
        gc.collect()
        torch.cuda.empty_cache()

detector_table = pd.DataFrame(rows).set_index(["detector", "text"])
detector_table.to_csv(os.path.join(RESULTS_DIR, "detector_summary.csv"))
detector_table

## What to do with the results

- **OCR:** if a model clearly beats the current pipeline on handwritten WER at a usable speed, it replaces Roboflow + TrOCR in `helper.transcribe_page` (`roadmap:ocr-bakeoff`). That also removes the external API and its key.
- **AI detection:** prefer the detector with the fewest humans flagged at an acceptable catch rate, especially on OCR-noised text. It replaces the classifier in `signals.create_ai_detection`, and its threshold goes in `HMAC/src/utils/score.js` (`roadmap:ai-detector-bakeoff`).
- `ocr_summary.csv` and `detector_summary.csv` are in your results folder. Keep them with the decision.